In [0]:
from pyspark.sql import functions as F
from pyspark.sql.types import DateType

# Generate date sequence from 1940-01-01 to 2030-12-31
df_dates = spark.sql("""
    SELECT explode(sequence(
        to_date('1940-01-01'), 
        to_date('2030-12-31'), 
        interval 1 day
    )) AS Date
""")

# Build comprehensive calendar columns
df_calendar = df_dates.select(
    F.date_format("Date", "yyyyMMdd").cast("int").alias("DateKey"),
    F.col("Date"),
    F.year("Date").alias("Year"),
    F.month("Date").alias("Month"),
    F.date_format("Date", "MMMM").alias("MonthName"),
    F.date_format("Date", "MMM").alias("MonthShortName"),
    F.dayofmonth("Date").alias("Day"),
    F.dayofweek("Date").alias("DayOfWeek"),
    F.date_format("Date", "EEEE").alias("DayName"),
    F.date_format("Date", "EEE").alias("DayShortName"),
    F.dayofyear("Date").alias("DayOfYear"),
    F.weekofyear("Date").alias("WeekOfYear"),
    F.ceil(F.dayofmonth("Date") / 7).cast("int").alias("WeekOfMonth"),
    F.quarter("Date").alias("Quarter"),
    F.date_format("Date", "yyyy-MM").alias("YearMonth"),
    F.concat(F.year("Date"), F.lit("-Q"), F.quarter("Date")).alias("YearQuarter"),
    F.when(F.dayofweek("Date").isin(1, 7), True).otherwise(False).alias("IsWeekend"),
    F.when(F.dayofweek("Date").isin(1, 7), False).otherwise(True).alias("IsWeekday"),
    F.when(
        (F.year("Date") % 4 == 0) & 
        ((F.year("Date") % 100 != 0) | (F.year("Date") % 400 == 0)),
        True
    ).otherwise(False).alias("IsLeapYear"),
    F.trunc("Date", "month").alias("StartOfMonth"),
    F.last_day("Date").alias("EndOfMonth"),
    F.date_sub(F.col("Date"), (F.dayofweek("Date") - 2 + 7) % 7).alias("StartOfWeek"),  # Monday
    F.date_add(F.date_sub(F.col("Date"), (F.dayofweek("Date") - 2 + 7) % 7), 6).alias("EndOfWeek"),  # Sunday
    F.trunc("Date", "year").alias("StartOfYear"),
    F.when(F.month("Date") <= 6, 
           F.concat(F.lit("H1-"), F.year("Date"))
    ).otherwise(
           F.concat(F.lit("H2-"), F.year("Date"))
    ).alias("HalfYear"),
    # Fiscal Year (assuming April start)
    F.when(F.month("Date") >= 4, F.year("Date") + 1).otherwise(F.year("Date")).alias("FiscalYear"),
    F.when(F.month("Date") >= 4, F.month("Date") - 3).otherwise(F.month("Date") + 9).alias("FiscalMonth"),
    F.when(F.month("Date").isin(4,5,6), 1)
     .when(F.month("Date").isin(7,8,9), 2)
     .when(F.month("Date").isin(10,11,12), 3)
     .otherwise(4).alias("FiscalQuarter")
)

# Write as a managed Delta table
df_calendar.write.format("delta") \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .saveAsTable("caresyncdatabricks.silver.calender")

print("Table caresyncdatabricks.silver.calender created successfully.")

In [0]:
%sql
select * from caresyncdatabricks.silver.calender limit 5